# Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Data Loading

In [ ]:
# Chemin vers le fichier
file_path = "/content/drive/MyDrive/Leh Kours/Master - Bioinformatique/Cours/M2 - S1/DEA - /V. BOURGEAIS/Projet/data/pancreas.h5ad"

# Chargement
adata = sc.read_h5ad(file_path)

# Afficher les dimensions
print(adata)

AnnData object with n_obs × n_vars = 16382 × 19093
    obs: 'tech', 'celltype', 'size_factors'
    layers: 'counts', None (.X)


# Kmeans

In [ ]:
# Application de K-means
kmeans = KMeans(
    n_clusters=4,
    init="k-means++",
    n_init=10,
    random_state=42
)

labels_kmeans = kmeans.fit_predict(adata)

# Visualisation en 2D avec PCA
X_2D = PCA(n_components=2).fit_transform(adata)

plt.scatter(
    X_2D[:, 0],
    X_2D[:, 1],
    c=labels_kmeans,
    cmap="viridis",
    s=15
)
plt.title("Clustering K-means")
plt.show()

# Louvain

In [ ]:
A = kneighbors_graph(
    X,
    n_neighbors=15,
    mode="connectivity",
    include_self=False
)

# Symétrisation du graphe
A = A.maximum(A.T)

# Conversion en graphe NetworkX
G = nx.from_scipy_sparse_array(A)

# --------------------------------
# 3. Application de Louvain
# --------------------------------

communities = nx.community.louvain_communities(
    G,
    weight=None,
    resolution=1.0,
    seed=42
)

# --------------------------------
# 4. Attribution des clusters
# --------------------------------

labels = np.zeros(X.shape[0], dtype=int)

for cluster_id, nodes in enumerate(communities):
    labels[list(nodes)] = cluster_id

print("Nombre de clusters :", len(communities))

# --------------------------------
# 5. Visualisation avec PCA
# --------------------------------

X_2D = PCA(n_components=2).fit_transform(X)

plt.figure(figsize=(8, 6))

plt.scatter(
    X_2D[:, 0],
    X_2D[:, 1],
    c=labels,
    cmap="viridis",
    s=15
)

plt.title("Clustering Louvain")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.colorbar(label="Cluster")
plt.show()

# Construction du graphe de voisinage
sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)

# Clustering Louvain
sc.tl.louvain(
    adata,
    resolution=1.0,
    key_added="louvain"
)

# Visualisation UMAP
sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color="louvain"
)

# Leiden

In [ ]:
import scanpy as sc
import matplotlib.pyplot as plt

# --------------------------------
# 1. Prétraitement des données
# --------------------------------

# Normalisation
sc.pp.normalize_total(adata, target_sum=1e4)

# Transformation logarithmique
sc.pp.log1p(adata)

# Sélection des gènes variables
sc.pp.highly_variable_genes(adata)

adata = adata[:, adata.var.highly_variable].copy()

# Standardisation
sc.pp.scale(adata, max_value=10)

# Analyse en composantes principales
sc.tl.pca(adata, n_comps=30)

# --------------------------------
# 2. Construction du graphe KNN
# --------------------------------

sc.pp.neighbors(
    adata,
    n_neighbors=15,
    n_pcs=30
)

# --------------------------------
# 3. Clustering Leiden
# --------------------------------

sc.tl.leiden(
    adata,
    resolution=1.0,
    key_added="leiden",
    random_state=42
)

# --------------------------------
# 4. Visualisation UMAP
# --------------------------------

sc.tl.umap(adata)

sc.pl.umap(
    adata,
    color="leiden",
    title="Clustering Leiden"
)

# --------------------------------
# 5. Nombre de clusters
# --------------------------------

print("Nombre de clusters :",
      adata.obs["leiden"].nunique())

print(adata.obs["leiden"].value_counts())

# Tester plusieurs resolutions :
for resolution in [0.1, 0.5, 1.0, 1.5, 2.0]:

    sc.tl.leiden(
        adata,
        resolution=resolution,
        key_added=f"leiden_{resolution}",
        random_state=42
    )

    n_clusters = adata.obs[
        f"leiden_{resolution}"
    ].nunique()

    print(f"Resolution {resolution} : {n_clusters} clusters")